## Image Analysis with Azure AI Content Understanding

This notebook uses **Azure AI Content Understanding** (the `azure-ai-contentunderstanding`
SDK) — one of Microsoft's recommended replacements for the deprecated Computer Vision
Image Analysis API. Content Understanding is a multimodal service that extracts structured
content from documents, images, audio, and video.

We use two prebuilt analyzers:

- **`prebuilt-imageSearch`** — returns a one-paragraph description (markdown + a `Summary`
  field) of a standalone image.
- **`prebuilt-read`** — performs OCR to read printed and handwritten text from an image.

> **Prerequisites:** A Microsoft Foundry resource in a region that supports Content
> Understanding. The `prebuilt-imageSearch` analyzer requires model deployments
> (`gpt-4.1-mini` and `text-embedding-3-large`) and a one-time default-model configuration
> on the resource. You also need the `Cognitive Services User` role on the resource.


### Installing Dependencies and Libraries

In [ ]:
%pip install azure-ai-contentunderstanding azure-identity pillow python-dotenv requests

### Setting Up the Environment

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()
# Endpoint looks like: https://<your-resource-name>.services.ai.azure.com/
CONTENTUNDERSTANDING_ENDPOINT = os.getenv("CONTENTUNDERSTANDING_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT')
TEXT_EMBEDDING_DEPLOYMENT = os.getenv('TEXT_EMBEDDING_DEPLOYMENT')

### Creating our Content Understanding Client

In [8]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential

# Authenticate with Entra ID (run `az login` first)
credential = DefaultAzureCredential()

client = ContentUnderstandingClient(endpoint=CONTENTUNDERSTANDING_ENDPOINT, credential=credential)

### Configure Default Model Deployments (one-time)

Prebuilt and custom analyzers need to know which of your Microsoft Foundry model deployments
to use. Run this once per resource (or whenever your deployment names change). The analyzers
in this notebook need a **completion** model and an **embedding** model, so make sure both are
deployed first.

### Define the Images to Analyze

The prebuilt `prebuilt-imageSearch` analyzer only returns a one-paragraph description.
To also get a **list of tags**, we create a custom analyzer (next section) with our own
field schema.

In [9]:
from azure.ai.contentunderstanding.models import AnalysisInput, AnalysisResult

images_to_analyze = [
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/building.jpg",
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/person.jpg",
    "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/street.jpg"
]

### Create a Custom Image Analyzer (Description + Tags)

Custom analyzers let us define our own field schema. Here we base it on `prebuilt-image`
and declare two generated fields: a `Description` string and a `Tags` array of strings.
Creating the analyzer is a one-time operation; re-running the cell reuses the existing one.

In [12]:
from azure.ai.contentunderstanding.models import (
    ContentAnalyzer,
    ContentFieldSchema,
    ContentFieldDefinition,
    ContentFieldType,
    GenerationMethod,
)
from azure.core.exceptions import ResourceExistsError

# analyzerId may not contain '-'
analyzer_id = "image_description_and_tags"

field_schema = ContentFieldSchema(
    name="image_tags_schema",
    description="Describe an image and list descriptive tags.",
    fields={
        "Description": ContentFieldDefinition(
            type=ContentFieldType.STRING,
            method=GenerationMethod.GENERATE,
            description="A one-sentence, gender-neutral description of the image.",
        ),
        "Tags": ContentFieldDefinition(
            type=ContentFieldType.ARRAY,
            method=GenerationMethod.GENERATE,
            description="A list of single word tags describing the visible content of the image.",
            item_definition=ContentFieldDefinition(type=ContentFieldType.STRING),
        ),
    },
)

analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-image",
    description="Custom image analyzer returning a description and a list of tags.",
    field_schema=field_schema,
    models={
        "completion": AZURE_OPENAI_DEPLOYMENT,
        "embedding": TEXT_EMBEDDING_DEPLOYMENT,
    },
)

try:
    client.begin_create_analyzer(analyzer_id=analyzer_id, resource=analyzer).result()
    print(f"Created custom analyzer '{analyzer_id}'.")
except ResourceExistsError:
    print(f"Custom analyzer '{analyzer_id}' already exists; reusing it.")

Created custom analyzer 'image_description_and_tags'.


### Get Image Analysis Results

In [13]:
for image_url in images_to_analyze:
    print(f"\nAnalyzing image: {image_url}\n")

    # Analyze with our custom analyzer; .result() waits for the operation to finish
    poller = client.begin_analyze(
        analyzer_id=analyzer_id,
        inputs=[AnalysisInput(url=image_url)],
    )
    result: AnalysisResult = poller.result()
    content = result.contents[0]

    fields = content.fields or {}

    description = fields.get("Description")
    if description is not None and getattr(description, "value", None):
        print("Description:")
        print(f"  {description.value}")

    # Tags is an array field; each element exposes its string via .value
    tags_field = fields.get("Tags")
    tags = [getattr(t, "value", t) for t in (tags_field.value or [])] if tags_field else []

    print("\nTags:")
    print("  " + ", ".join(tags) if tags else "  (none)")


Analyzing image: https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/building.jpg

Description:
  A large domed government building stands behind a wide green lawn under a partly cloudy sky, with a few people walking in the foreground.

Tags:
  capitol, dome, government, building, architecture, landmark, lawn, grass, trees, sky, clouds, daytime, people, steps, columns, flag

Analyzing image: https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/person.jpg

Description:
  A person in a gray blazer and glasses gestures while speaking on a stage with a four-color square logo in the background.

Tags:
  speaker, stage, presentation, blazer, glasses, bald, gesture, microphone, logo, Microsoft, technology, conference, keynote, backdrop, lighting

Analyzing image: https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/street.jpg

Descriptio

### Read Text from an Image with the prebuilt-read Analyzer

The `prebuilt-read` analyzer performs OCR and returns the recognized text as markdown.
It handles both printed and handwritten text.

![handwritten.jpg](./Images/handwritten.png)

In [14]:
from PIL import Image
from io import BytesIO
import requests

ocr_image_url = "https://raw.githubusercontent.com/kuljotSB/AI-102-Certification/main/AzureAIVision/Image-Analysis/Images/handwritten.png"

# Download the image so we can display it alongside the transcription
response = requests.get(ocr_image_url)
original_image = Image.open(BytesIO(response.content)).convert("RGB")

# Run OCR with the prebuilt-read analyzer
poller = client.begin_analyze(
    analyzer_id="prebuilt-read",
    inputs=[AnalysisInput(url=ocr_image_url)],
)
result: AnalysisResult = poller.result()

content = result.contents[0]

print("Transcribed text:\n")
print(content.markdown)


Transcribed text:

Hello,

Simply Noted has developed
incredible proprietary robotic
technology to write your message
and envelopes with a genuine
real pen. It is completely
indistinguishable from a humans
handwriting.

Try us today!
Simply Noted

